In [6]:
import os
import json
from textwrap import dedent

DPO_DATASET_DIR = "./finetuning_data_dpo/crm-dpo-dataset/"
SFT_DATASET_DIR = "./finetuning_data_dpo/crm-sft-dataset/"

# 기존 DPO 데이터셋 파일 읽기
with open(os.path.join(DPO_DATASET_DIR, "cycle_01_v2.json"), "r") as f:
    data = json.load(f)
    print(f"로드된 DPO 데이터 수 : {len(data)}")
    print(f"DPO 데이터 컬럼 들 : {list(data[0].keys())}")
    print(data[:10])
    print(type(data), type(f))

# SFT용 추가 프롬프트 정의
output_template = dedent("""
[출력 규칙]
- 불필요한 요소를 제거하고 핵심 문장만 포함한다.
- 의미 전달에 필요한 핵심 정보만 간결하게 작성한다.
- 다음 요소는 포함하지 않는다:
  1) 영어/한국어의 어색한 혼용 (브랜드/제품 고유명 제외, 예: 'everyday 사용')
  2) 페르소나/개인정보의 직접 호명 (예: 'Budget_Seeker님')
  3) 과도한 특수문자, 이모지, 구분선
""")

os.makedirs(SFT_DATASET_DIR, exist_ok=True)
with open(os.path.join(SFT_DATASET_DIR, "cycle_01.jsonl"), "w") as f:
    for example in data:
        prompt = example.get("prompt", "")
        chosen = example.get("chosen", "")
        prompt = "다음 조건에 맞는 CRM 메시지를 작성하세요. " + prompt + f" {output_template.replace("\n", " ")}"
        f.write(json.dumps({"prompt":prompt, "chosen":chosen}, ensure_ascii=False) + "\n")

with open(os.path.join(SFT_DATASET_DIR, "cycle_01.jsonl"), "r") as f:
    print(type(f))
    print(f"저장된 SFT 데이터 수 : {sum(1 for _ in f)}")

로드된 DPO 데이터 수 : 1248
DPO 데이터 컬럼 들 : ['prompt', 'chosen', 'rejected', 'best_index', 'rejected_index', 'reason_best', 'reason_rejected']
[{'prompt': '[컨텍스트]\n- Persona: Budget_Seeker (가격 대비 효율, 실사용 후기, 복합성/지성, 가벼운 트러블, 가성비 중심, 쿠폰·세일 적극 활용, 브랜드 충성도 낮음)\n- Stage: Referral\n- Brand/Product: 에뛰드 / 뽀오얀 미소 발효 립&아이 리무버 250ml (대용량)\n- Price: 12,750원\n- Event: 없음', 'chosen': '```json\n{\n  "title": "[뽀오얀 미소 발효 립&아이 리무버] efficacy & affordable 선택 가이드",\n  "body": "EDDY Range의 포어 발효 기술로 눈 clarity 유지가 핵심입니다! 대용량 250ml로 everyday 사용부터 여행까지 편리하게 쓸 수 있어요. 지성/복합성 피부도 부담 없이 사용 가능하고, 기존 제품 대비 비용 대비 성능이 뛰어나다는 후기 다수가 있습니다. 오늘부터 10일간만 특가로 판매 중이니 서둘러 보세요! 함께 써줄 친구에게 추천해주면 추가 혜택도 받을 수 있답니다."\n}\n```', 'rejected': '```json\n{\n  "title": "⌛ 혜택 종료 전 마지막 추천 기회",\n  "body": "[브랜드명]에서 추천해준 Olive Kit이 정말 좋아요! 가볍지만 효과적인 입술&눈 리무버로 하루 종일 사용하실 수 있어요. 여러 번 쓸 수 있어 부담 없이 사용할 수 있답니다. 오늘도 같이 써볼까요? 👇"\n}\n```', 'best_index': 2, 'rejected_index': 0, 'reason_best': '가장 간결하고 필요한 정보만 담아 브랜드명 노출 최소화, 페르소나 호명 없이도 메시지의 가치와 혜택이 명확히 전달된